# CSE 151B — Prompt-Engineering-Only Pipeline (Qwen3-4B-Thinking)

A **simple, single-model** notebook that pushes accuracy using *only* inference-time
prompt engineering — no fine-tuning. Same data loading and `.venv` as the starter.

The accuracy levers, in order of impact:

1. **Self-consistency** — draw `N_SAMPLES` independent chain-of-thought samples per
   question and majority-vote the final answer. This is the single biggest gain you
   can get from prompting alone.
2. **Format-aware system prompts** tuned to how the official `Judger` grades:
   - Answers are extracted from `\boxed{}` *after* the last `</think>`.
   - Numeric matching is **relative 1e-8** → rounded decimals fail. Demand **exact
     closed-form** answers (`\frac{1}{3}`, `\sqrt{2}`, `\pi`), never approximations.
   - **Multi-`[ANS]`** questions: all sub-answers must be correct, matched in order.
   - Some free-form `[ANS]` slots are followed by inline lettered choices
     (`A. ... B. ...`) — the answer for those is the **letter only**, not the text.
   - No thousands-separator commas inside numbers (`39746`, not `39,746`).
3. **Robust extraction + voting** that mirrors the judger's contiguous-`\boxed{}`
   logic, then emits a single canonical `\boxed{}` so the graded answer is exactly
   the voted one — while keeping the full reasoning trace in the submission.

The pipeline is **checkpointed**: generation appends raw traces to
`results/prompt_eng_samples.jsonl`, so a crash/restart resumes where it left off.

## 0. Environment

Reuses the project `.venv` (identical to the starter). All packages are already
installed there; uncomment the pip line only on a fresh machine.

In [1]:
# One-time install on a fresh machine (uncomment):
# !.venv/bin/python -m pip install -q sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1
!source ./.venv/bin/activate && echo "venv ready"

venv ready


## 1. Configuration

Everything you might tune lives here.

- `TARGET` — `"private"` builds the leaderboard submission; `"public"` validates
  against ground truth so you can measure accuracy locally.
- `N_SAMPLES` — self-consistency width. Higher = better accuracy, linearly more
  compute. `8` is a strong default; drop to `3`–`4` for a quick pass.
- `QUANTIZATION` — `None` runs BF16 (best quality, needs ~10 GB VRAM for the 4B
  model; fine on this box). Set to `"bitsandbytes"` for INT8 if VRAM-constrained
  (matches the starter).

In [ ]:
import os, json, re, csv, sys, gc
from pathlib import Path
from collections import Counter, defaultdict

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID   = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

PUBLIC_PATH  = "data/public.jsonl"
PRIVATE_PATH = "data/private.jsonl"

TARGET = "private"                            # "private" (submit) or "public" (validate)

RESULTS_DIR    = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)
SAMPLES_PATH   = RESULTS_DIR / f"prompt_eng_samples_{TARGET}.jsonl"
SUBMISSION_CSV = "submission_prompt_eng.csv"

# --- Adaptive self-consistency ---
# Pass 1: generate INITIAL_SAMPLES for every question.
# Pass 2: only questions where the top-answer fraction < AGREEMENT_THRESH
#         receive EXTRA_SAMPLES more (capped at MAX_SAMPLES total).
# Easy questions where all samples agree stop after pass 1, saving ~50% of compute.
INITIAL_SAMPLES  = 4     # first-pass width
MAX_SAMPLES      = 8     # ceiling for uncertain questions
AGREEMENT_THRESH = 1.0   # 1.0 = all samples must agree to stop early
EXTRA_SAMPLES    = MAX_SAMPLES - INITIAL_SAMPLES

N_SAMPLES  = MAX_SAMPLES   # alias so the SamplingParams cell below still works
CHUNK_SIZE = 32            # questions per generate() call

# --- Generation budget ---
QUANTIZATION  = None
MAX_MODEL_LEN = 32768
MAX_TOKENS    = 2048
GPU_MEM_UTIL  = 0.85

# --- Qwen3-Thinking recommended sampling (do NOT use greedy) ---
TEMPERATURE, TOP_P, TOP_K, MIN_P = 0.6, 0.95, 20, 0.0

print(f"TARGET={TARGET}  INITIAL={INITIAL_SAMPLES}  MAX={MAX_SAMPLES}  "
      f"AGREE_THRESH={AGREEMENT_THRESH}  CHUNK={CHUNK_SIZE}  quant={QUANTIZATION}")

TARGET=private  INITIAL=4  MAX=8  AGREE_THRESH=1.0  CHUNK=16  quant=None


## 2. Load data + judger

In [3]:
sys.path.insert(0, ".")
from judger import Judger
from utils import last_boxed_only_string, remove_boxed
judger = Judger(strict_extract=False)

def load_jsonl(p):
    return [json.loads(l) for l in open(p)]

public_data  = load_jsonl(PUBLIC_PATH)
private_data = load_jsonl(PRIVATE_PATH)
data = private_data if TARGET == "private" else public_data
n_mcq = sum(bool(d.get("options")) for d in data)
print(f"public={len(public_data)}  private={len(private_data)}")
print(f"running on '{TARGET}': {len(data)} questions ({n_mcq} MCQ, {len(data)-n_mcq} free-form)")

public=1126  private=943
running on 'private': 943 questions (300 MCQ, 643 free-form)


## 3. Prompt construction (format-aware CoT)

Zero-shot CoT — the Thinking model reasons natively inside `<think>` tags, so
few-shot mostly adds noise and token cost. The system prompt is doing the heavy
lifting: it spells out exactly the answer format the `Judger` rewards.

In [4]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem with careful, rigorous, "
    "step-by-step reasoning.\n"
    "ANSWER FORMAT (your answer is graded by exact match — follow precisely):\n"
    "- Finish your response with the final answer(s) inside a single \\boxed{...}.\n"
    "- Give EXACT, closed-form values: keep fractions (\\frac{1}{3}), radicals "
    "(\\sqrt{2}), and constants (\\pi, e) symbolic. Do NOT round — a decimal "
    "approximation of an exact value is graded WRONG. If only a decimal is "
    "possible, give at least 10 significant figures.\n"
    "- Write plain numbers with no thousands separators: 39746, never 39,746.\n"
    "- Put ONLY the value(s) in the box: no units, words, \\text{}, or explanation "
    "unless the question explicitly asks for them.\n"
    "- Multiple [ANS] slots: output every answer in order, comma-separated, inside "
    "the SAME box, e.g. \\boxed{3, 7, -1}. Give exactly one value per [ANS], in the "
    "order the slots appear.\n"
    "- Choice slots: if an [ANS] is followed by inline lettered options "
    "(e.g. 'A. ...  B. ...'), the answer for that slot is the single capital LETTER "
    "of the correct option, NOT the option text, e.g. \\boxed{-1.46, B}."
    "Keep the solution concise. End with the final answer in \\boxed{}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. Read the problem and the lettered answer "
    "choices, reason step-by-step, then choose the single best option. "
    "Output ONLY that option's capital letter inside \\boxed{}, e.g. \\boxed{C}."
    "Keep the solution concise. End with the final answer in \\boxed{}."
)

def _format_choices(options):
    labels = [chr(65 + i) for i in range(len(options))]
    return "\n".join(f"{l}. {o.strip()}" for l, o in zip(labels, options))

def build_messages(item):
    is_mcq = bool(item.get("options"))
    system = SYSTEM_PROMPT_MCQ if is_mcq else SYSTEM_PROMPT_MATH
    parts = [item["question"].strip()]
    if is_mcq:
        parts.append("\nOptions:\n" + _format_choices(item["options"]))
        parts.append("\nReason step-by-step, then give the chosen letter in \\boxed{}.")
    else:
        n = item["question"].count("[ANS]")
        if n >= 2:
            parts.append(f"\nThis question has {n} [ANS] slots. Give all {n} answers "
                         "in order, comma-separated, in one \\boxed{}.")
    return [{"role": "system", "content": system},
            {"role": "user",   "content": "\n".join(parts)}]

def chat_template_prompt(tok, messages):
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# sanity preview
print(build_messages(next(d for d in data if d.get("options")))[1]["content"][:200], "\n---")
print(build_messages(next(d for d in data if not d.get("options")))[1]["content"][:200])

Assuming the weights corresponding to the sign values are reduced by 1/10, then the arithmetic mean is ().

Options:
A. Unchanged
B. Increased by ten percent
C. Reduced by one percent
D. Increased by  
---
Use the order of operations to simplify: a) $[13-(11-11)]-[8-(5-6)]=$ [ANS]
b) $4 \cdot 3-2+2 \cdot 3=$ [ANS]

This question has 2 [ANS] slots. Give all 2 answers in order, comma-separated, in one \bo


## 4. Answer extraction + voting

These mirror the judger's behaviour: strip everything up to the last `</think>`,
read `\boxed{}` contents in order, reconcile them to the number of `[ANS]` slots,
then majority-vote **per slot**. `finalize_response` keeps the chosen full trace
and appends one canonical `\boxed{}` so the judge extracts exactly the voted
answer (multi-answer must be one comma-box or a contiguous group of boxes).

In [5]:
_LETTER_RE = re.compile(r"\\boxed\{\s*([A-Za-z])\s*\}")

def extract_letter(text):
    m = _LETTER_RE.search(text or "")
    if m: return m.group(1).upper()
    for line in reversed((text or "").splitlines()):
        ms = re.findall(r"\b([A-Z])\b", line.strip())
        if ms: return ms[-1]
    return ""

def _strip_thinking(text):
    if not text: return ""
    j = text.rfind("</think>")
    return text[j + len("</think>"):] if j >= 0 else text

def _all_boxed_in_order(text):
    out, i, needle = [], 0, "\\boxed{"
    while True:
        j = text.find(needle, i)
        if j < 0: break
        k = j + len(needle); depth = 1
        while k < len(text) and depth > 0:
            if text[k] == '{': depth += 1
            elif text[k] == '}': depth -= 1
            k += 1
        if depth == 0:
            out.append(text[j + len(needle):k - 1].strip())
        i = k
    return out

def n_ans_slots(item):
    if item.get("options"): return 1
    n = item.get("question", "").count("[ANS]")
    return n if n >= 1 else 1

def extract_free_answers(text, n_slots):
    post = _strip_thinking(text) or text
    boxes = _all_boxed_in_order(post) or _all_boxed_in_order(text)
    if not boxes: return None
    if n_slots == 1: return [boxes[-1]]
    if len(boxes) == 1:                                   # one comma-box
        parts = judger.split_by_comma(boxes[0])
        return [p.strip() for p in parts] if len(parts) == n_slots else None
    if len(boxes) == n_slots: return boxes                # one box per slot
    if len(boxes) > n_slots:  return boxes[-n_slots:]     # extra (per-step) boxes -> keep last n
    expanded = []                                         # fewer boxes -> try expanding comma-boxes
    for b in boxes:
        parts = judger.split_by_comma(b)
        expanded.extend(p.strip() for p in parts) if len(parts) > 1 else expanded.append(b)
    if len(expanded) == n_slots: return expanded
    if len(expanded) > n_slots:  return expanded[-n_slots:]
    return None

def _vote_key(s):
    """Canonicalize one answer for vote bucketing (numeric-aware)."""
    if s is None: return ""
    t = s.strip().strip("$").strip()
    if not t: return ""
    try:
        v = float(t.replace(",", "")); return "0" if v == 0 else f"{v:.6g}"
    except Exception: pass
    try:
        v = float(judger.norm_math_str(t)); return "0" if v == 0 else f"{v:.6g}"
    except Exception: pass
    try: return judger.norm_math_str(t)
    except Exception: return t

def canon_keys(item, trace):
    if item.get("options"):
        l = extract_letter(trace); return [l] if l else None
    ans = extract_free_answers(trace, n_ans_slots(item))
    return [_vote_key(a) for a in ans] if ans is not None else None

def summarize(item, traces):
    """Majority-vote over traces. Returns {is_mcq, box, frac, win_keys}."""
    is_mcq = bool(item.get("options"))
    if is_mcq:
        c = Counter()
        for t in traces:
            l = extract_letter(t)
            if l: c[l] += 1
        if not c: return {"is_mcq": True, "box": "", "frac": 0.0, "win_keys": None}
        top, cnt = c.most_common(1)[0]
        return {"is_mcq": True, "box": top, "frac": cnt / len(traces), "win_keys": [top]}
    n = n_ans_slots(item)
    per = [Counter() for _ in range(n)]; rep = [dict() for _ in range(n)]; used = 0
    for t in traces:
        ans = extract_free_answers(t, n)
        if ans is None or len(ans) != n: continue
        used += 1
        for i, a in enumerate(ans):
            k = _vote_key(a)
            if not k: continue
            per[i][k] += 1
            if k not in rep[i] or len(a) < len(rep[i][k]): rep[i][k] = a
    if used == 0: return {"is_mcq": False, "box": "", "frac": 0.0, "win_keys": None}
    parts, fracs, wk = [], [], []
    for i in range(n):
        if not per[i]: return {"is_mcq": False, "box": "", "frac": 0.0, "win_keys": None}
        tk, tn = per[i].most_common(1)[0]
        wk.append(tk); parts.append(rep[i][tk]); fracs.append(tn / used)
    return {"is_mcq": False, "box": ", ".join(p.strip() for p in parts),
            "frac": min(fracs), "win_keys": wk}

def finalize_response(item, traces, summary):
    """Full reasoning trace + a canonical final \\boxed{} equal to the vote."""
    box, win = summary["box"], summary["win_keys"]
    base = ""
    if win:                                   # prefer reasoning that reached the voted answer
        for t in traces:
            if canon_keys(item, t) == win: base = t; break
    if not base:
        base = max(traces, key=len) if traces else ""
    if box:
        return (base + "\n\nFinal answer: \\boxed{" + box + "}").strip()
    return base.strip() or "\\boxed{}"

def judge_response(item, response):
    if bool(item.get("options")):
        return extract_letter(response) == str(item["answer"]).strip().upper()
    gold = item["answer"] if isinstance(item["answer"], list) else [item["answer"]]
    try:
        return bool(judger.auto_judge(pred=response, gold=gold, options=[[]] * len(gold)))
    except Exception:
        return False

print("extraction + voting helpers ready")

extraction + voting helpers ready


## 5. Load the model (vLLM)

`enable_prefix_caching=True` reuses the shared system-prompt KV across questions,
and `SamplingParams(n=N_SAMPLES)` draws all self-consistency samples for a prompt
in one efficient batched pass.

In [6]:
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

llm_kwargs = dict(
    model=MODEL_ID,
    enable_prefix_caching=True,
    gpu_memory_utilization=GPU_MEM_UTIL,
    max_model_len=MAX_MODEL_LEN,
    trust_remote_code=True,
    max_num_seqs=128,
    enforce_eager=True,   # no CUDA toolkit on this box (no nvcc) -> skip torch.compile/CUDA-graph capture
)
if QUANTIZATION == "bitsandbytes":
    llm_kwargs.update(quantization="bitsandbytes", load_format="bitsandbytes")

llm = LLM(**llm_kwargs)

sampling_params = SamplingParams(
    n=N_SAMPLES, max_tokens=MAX_TOKENS,
    temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K, min_p=MIN_P,
)
print("Model loaded.")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.8k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

(EngineCore pid=1282) INFO 05-28 17:05:39 [weight_utils.py:619] Time spent downloading weights for Qwen/Qwen3-4B-Thinking-2507: 11.346546 seconds
(EngineCore pid=1282) INFO 05-28 17:05:40 [weight_utils.py:938] Filesystem type for checkpoints: OVERLAY. Checkpoint size: 7.49 GiB. Available RAM: 351.93 GiB.
(EngineCore pid=1282) INFO 05-28 17:05:40 [weight_utils.py:961] Auto-prefetch is disabled because the filesystem (OVERLAY) is not a recognized network FS (NFS/Lustre). If you want to force prefetching, start vLLM with --safetensors-load-strategy=prefetch.


Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


(EngineCore pid=1282) INFO 05-28 17:05:59 [default_loader.py:397] Loading weights took 19.21 seconds
(EngineCore pid=1282) INFO 05-28 17:06:00 [gpu_model_runner.py:4959] Model loading took 7.64 GiB memory and 35.807397 seconds
(EngineCore pid=1282) INFO 05-28 17:06:15 [backends.py:1089] Using cache directory: /tmp/xdg-cache/vllm/torch_compile_cache/2e4988a16e/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=1282) INFO 05-28 17:06:15 [backends.py:1148] Dynamo bytecode transform time: 14.36 s
(EngineCore pid=1282) INFO 05-28 17:06:25 [backends.py:378] Cache the graph of compile range (1, 8192) for later use
(EngineCore pid=1282) INFO 05-28 17:06:30 [backends.py:393] Compiling a graph for compile range (1, 8192) takes 15.63 s
(EngineCore pid=1282) INFO 05-28 17:06:34 [decorators.py:708] saved AOT compiled function to /tmp/xdg-cache/vllm/torch_compile_cache/torch_aot_compile/b6b57280594ce60c1f81f5baed2604e62e92ddc619625713744ae07a0a033f36/rank_0_0/model
(EngineCore pid=1282) INFO

(EngineCore pid=1282) Process EngineCore:
(EngineCore pid=1282) Traceback (most recent call last):
(EngineCore pid=1282)   File "/home/prgowda/.local/share/uv/python/cpython-3.14.5-linux-x86_64-gnu/lib/python3.14/multiprocessing/process.py", line 320, in _bootstrap
(EngineCore pid=1282)     self.run()
(EngineCore pid=1282)     ~~~~~~~~^^
(EngineCore pid=1282)   File "/home/prgowda/.local/share/uv/python/cpython-3.14.5-linux-x86_64-gnu/lib/python3.14/multiprocessing/process.py", line 108, in run
(EngineCore pid=1282)     self._target(*self._args, **self._kwargs)
(EngineCore pid=1282)     ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(EngineCore pid=1282)   File "/home/prgowda/private/151B_SP26_Competition-main/.venv/lib/python3.14/site-packages/vllm/v1/engine/core.py", line 1144, in run_engine_core
(EngineCore pid=1282)     raise e
(EngineCore pid=1282)   File "/home/prgowda/private/151B_SP26_Competition-main/.venv/lib/python3.14/site-packages/vllm/v1/engine/core.py", line 1114, in run_engi

RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}

## 6. Adaptive self-consistency generation (checkpointed)

**Pass 1** — generate `INITIAL_SAMPLES` traces for every question and flush to disk.

**Agreement check** — for each question, compute the top-answer vote fraction.
If it equals `AGREEMENT_THRESH` (default 1.0 = unanimous), the question is done.
Otherwise it goes into Pass 2.

**Pass 2** — generate `EXTRA_SAMPLES` more traces only for uncertain questions,
then re-vote using all samples combined.

The checkpoint file (`SAMPLES_PATH`) is append-only, so re-running this cell after
a crash resumes exactly where it left off — both passes are resumable.

In [ ]:
import time

# ── helpers ──────────────────────────────────────────────────────────────────

def load_done(path):
    by_id = defaultdict(list)
    if path.exists():
        with open(path) as f:
            for line in f:
                try:
                    r = json.loads(line); by_id[r["id"]].append(r["trace"])
                except Exception: pass
    return by_id

def is_confident(item, traces):
    """True if the top-answer fraction across existing traces >= AGREEMENT_THRESH."""
    if len(traces) == 0:
        return False
    s = summarize(item, traces)
    return s["frac"] >= AGREEMENT_THRESH and s["box"] != ""

def run_generation_pass(questions, n_new, pass_label):
    """
    Generate n_new new samples for each question in `questions`.
    Appends results to SAMPLES_PATH and updates the global by_id dict.
    """
    total        = len(questions)
    total_chunks = (total + CHUNK_SIZE - 1) // CHUNK_SIZE
    sp = SamplingParams(
        n=n_new, max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K, min_p=MIN_P,
    )

    print(f"\n{'='*60}")
    print(f"{pass_label}: {total} questions × {n_new} samples = "
          f"{total * n_new} sequences  |  {total_chunks} chunks of {CHUNK_SIZE}")
    print(f"{'='*60}", flush=True)

    pass_start = time.time()

    with open(SAMPLES_PATH, "a") as out:
        for s in range(0, total, CHUNK_SIZE):
            chunk_start = time.time()
            chunk       = questions[s : s + CHUNK_SIZE]
            chunk_idx   = s // CHUNK_SIZE + 1

            print(f"\n[{pass_label} | Chunk {chunk_idx}/{total_chunks}] "
                  f"Preparing {len(chunk)} prompts...", flush=True)
            prompts = [chat_template_prompt(tokenizer, build_messages(it)) for it in chunk]

            print(f"[{pass_label} | Chunk {chunk_idx}/{total_chunks}] "
                  f"Sending {len(chunk) * n_new} sequences to vLLM "
                  f"({len(chunk)} prompts × {n_new} samples)...", flush=True)
            outs = llm.generate(prompts, sp, use_tqdm=True)

            print(f"[{pass_label} | Chunk {chunk_idx}/{total_chunks}] "
                  f"Writing results to disk...", flush=True)
            for it, o in zip(chunk, outs):
                for comp in o.outputs:
                    trace = comp.text.strip()
                    out.write(json.dumps({"id": it["id"], "trace": trace}) + "\n")
                    by_id[it["id"]].append(trace)
            out.flush()

            elapsed    = time.time() - chunk_start
            done_so_far = min(s + CHUNK_SIZE, total)
            pct         = 100 * done_so_far / total
            avg_per_q   = (time.time() - pass_start) / done_so_far
            eta_s       = int(avg_per_q * (total - done_so_far))
            print(f"[{pass_label} | Chunk {chunk_idx}/{total_chunks}] "
                  f"Chunk done in {elapsed:.1f}s  |  "
                  f"{done_so_far}/{total} questions ({pct:.1f}%)  |  "
                  f"ETA {eta_s // 60}m {eta_s % 60:02d}s", flush=True)

    pass_time = time.time() - pass_start
    print(f"\n{pass_label} complete in {pass_time/60:.1f} min.", flush=True)

# ── load checkpoint ───────────────────────────────────────────────────────────

by_id = load_done(SAMPLES_PATH)
print(f"Loaded checkpoint: {sum(len(v) for v in by_id.values())} existing traces "
      f"across {len(by_id)} questions.")

# ── PASS 1: initial samples for every question ────────────────────────────────

pass1_todo = [it for it in data if len(by_id.get(it["id"], [])) < INITIAL_SAMPLES]
already    = len(data) - len(pass1_todo)
print(f"\n{already}/{len(data)} questions already have ≥{INITIAL_SAMPLES} samples (skipping).")

if pass1_todo:
    run_generation_pass(pass1_todo, INITIAL_SAMPLES, "Pass 1")
else:
    print("Pass 1: nothing to do.")

# ── agreement check ───────────────────────────────────────────────────────────

confident = [it for it in data if     is_confident(it, by_id.get(it["id"], []))]
uncertain = [it for it in data if not is_confident(it, by_id.get(it["id"], []))
                                  and len(by_id.get(it["id"], [])) < MAX_SAMPLES]

print(f"\n── Agreement check (thresh = {AGREEMENT_THRESH}) ──────────────────────────")
print(f"  Confident (stopping early) : {len(confident):4d} / {len(data)}")
print(f"  Uncertain (need more)      : {len(uncertain):4d} / {len(data)}")
if uncertain:
    pct_uncertain = 100 * len(uncertain) / len(data)
    saved = 100 * len(confident) / len(data)
    print(f"  → generating {EXTRA_SAMPLES} more samples for {pct_uncertain:.1f}% of questions "
          f"(saved {saved:.1f}% of pass-2 work by stopping early)")

# ── PASS 2: extra samples for uncertain questions only ────────────────────────

if uncertain:
    run_generation_pass(uncertain, EXTRA_SAMPLES, "Pass 2")
else:
    print("\nPass 2: all questions are confident — nothing to do.")

# ── summary ───────────────────────────────────────────────────────────────────

total_traces = sum(len(v) for v in by_id.values())
print(f"\n{'='*60}")
print(f"Adaptive generation complete.")
print(f"Total traces: {total_traces}  |  Avg per question: {total_traces / len(data):.2f}")

Loaded checkpoint: 32 existing traces across 4 questions.

4/943 questions already have ≥4 samples (skipping).

Pass 1: 939 questions × 4 samples = 3756 sequences  |  59 chunks of 16

[Pass 1 | Chunk 1/59] Preparing 16 prompts...
[Pass 1 | Chunk 1/59] Sending 64 sequences to vLLM (16 prompts × 4 samples)...


Rendering prompts:   0%|          | 0/16 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/64 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

[Pass 1 | Chunk 1/59] Writing results to disk...
[Pass 1 | Chunk 1/59] Chunk done in 5130.5s  |  16/939 questions (1.7%)  |  ETA 4932m 44s

[Pass 1 | Chunk 2/59] Preparing 16 prompts...
[Pass 1 | Chunk 2/59] Sending 64 sequences to vLLM (16 prompts × 4 samples)...


Rendering prompts:   0%|          | 0/16 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/64 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

## 7. Vote and write the submission

For every question we majority-vote the answer and write the full winning trace
(plus the canonical `\boxed{}`) to the CSV, properly quoted/escaped.

In [ ]:
rows = []
empty = low_conf = 0
for item in data:
    traces  = by_id.get(item["id"], [])
    summary = summarize(item, traces)
    resp    = finalize_response(item, traces, summary)
    rows.append((item["id"], resp))
    if not summary["box"]: empty += 1
    elif summary["frac"] < 0.5: low_conf += 1

with open(SUBMISSION_CSV, "w", newline="") as f:
    w = csv.writer(f, quoting=csv.QUOTE_ALL)
    w.writerow(["id", "response"])
    for qid, resp in rows:
        w.writerow([qid, resp])

assert len(rows) == len(data), "row count must match the test set"
print(f"Wrote {SUBMISSION_CSV}: {len(rows)} rows "
      f"({empty} with no parseable answer, {low_conf} low-confidence < 0.5)")

## 8. Local accuracy check (only meaningful when `TARGET="public"`)

Scores the finalized responses against ground truth, broken down by question type.
Skipped automatically on the private set (no answers available).

In [ ]:
if TARGET == "public":
    def acc(sub): return 100 * sum(sub) / len(sub) if sub else 0.0
    mcq, free = [], []
    for item in public_data:
        traces = by_id.get(item["id"], [])
        if not traces: continue
        resp = finalize_response(item, traces, summarize(item, traces))
        ok = judge_response(item, resp)
        (mcq if item.get("options") else free).append(ok)
    print("=" * 46)
    print(f"  MCQ       : {sum(mcq):4d} / {len(mcq):4d}  ({acc(mcq):.2f}%)")
    print(f"  Free-form : {sum(free):4d} / {len(free):4d}  ({acc(free):.2f}%)")
    allr = mcq + free
    print(f"  Overall   : {sum(allr):4d} / {len(allr):4d}  ({acc(allr):.2f}%)")
    print("=" * 46)
else:
    print("TARGET='private' — no ground truth. Set TARGET='public' to measure accuracy.")